# Starter kit — Hackathon Mines Paris 2026 — Analyse d'incidents PV / éolien

## Objectif du hackathon

Construire un pipeline agentique qui, à partir des rapports d'incidents PV et éolien `data/incidents_students.csv`, réalise :
- du **topic modeling** (regroupement des incidents par thème),
- une **analyse de causes**,
- des **statistiques globales** (par site, sévérité, saisonalité, durée d'arrêt, coût...),
- des **propositions de remédiation** (maintenance, évolutions techniques).

Ce notebook couvre la première brique, le topic modeling, et sert de base pour construire le reste
du pipeline.

## 1. Configuration

In [1]:
import os

# Réseau d'entreprise avec inspection SSL : utiliser le magasin de certificats de l'OS
# (doit être appelé avant la création de tout client HTTP).
import truststore

truststore.inject_into_ssl()

from dotenv import load_dotenv

# override=True : le .env prime sur d'éventuelles variables déjà présentes dans le kernel
load_dotenv(override=True)

AZURE_OPENAI_API_KEY = os.environ["AZURE_OPENAI_API_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_API_VERSION = os.environ["AZURE_OPENAI_API_VERSION"]
EMBEDDING_DEPLOYMENT = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
LLM_DEPLOYMENT = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT"]

# Vérification de la config lue (la clé n'est jamais affichée)
print("endpoint   :", repr(AZURE_OPENAI_ENDPOINT))
print("api_version:", repr(AZURE_OPENAI_API_VERSION))
print("embedding  :", repr(EMBEDDING_DEPLOYMENT))
print("chat       :", repr(LLM_DEPLOYMENT))
if "/api/projects" in AZURE_OPENAI_ENDPOINT:
    print(
        "⚠️ L'endpoint contient /api/projects : AzureOpenAI attend l'endpoint de la ressource, "
        "ex. https://<ressource>.cognitiveservices.azure.com/"
    )

endpoint   : 'https://hackathon-mines-paris-2026-foundry-resource.services.ai.azure.com/'
api_version: '2024-12-01-preview'
embedding  : 'text-embedding-3-small-4-d0dTdW'
chat       : 'gpt-4.1-nano-4-d0dTdW'


In [2]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# Test de connexion : isole un problème de config (endpoint / version / déploiement)
# avant de lancer le calcul des embeddings sur tout le jeu de données.
_probe = client.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=["test"])
print("Connexion OK, dimension des embeddings :", len(_probe.data[0].embedding))

Connexion OK, dimension des embeddings : 1536


## 2. Charger les données

Les rapports sont fournis dans 3 formats (`document_format`) :
- `text` : texte déjà disponible dans la colonne `text` du CSV.
- `pdf_digital` : PDF avec texte extractible (`document_path` pointe vers `data/pdf_digital/`).
- `pdf_scan` : PDF scanné (image), nécessitant de l'OCR (`document_path` pointe vers
  `data/pdf_scan/`).

Pour ces deux derniers formats, seules les colonnes `site`, `document_format` et `document_path`
sont renseignées dans le CSV : à vous d'extraire le texte (et le reste des métadonnées) des PDF.

Ce starter kit se limite volontairement au sous-ensemble `text`, déjà prêt à l'emploi, pour
démarrer le topic modeling. 

In [ ]:
import pandas as pd
import base64, json, os, re, time
import concurrent.futures as cf
import pymupdf
from openai import RateLimitError
from tqdm.auto import tqdm

df_all = pd.read_csv("data/incidents_students.csv")
print(df_all["document_format"].value_counts())

df = df_all[df_all["document_format"] == "text"].reset_index(drop=True)
print(df.shape)
df.head()

document_format
pdf_scan       600
text           600
pdf_digital    600
Name: count, dtype: int64
(600, 13)


,incident_id,site,asset_type,equipment_id,date,severity,downtime_hours,estimated_cost_eur,weather_condition,technician_id,text,document_format,document_path
0,2,Parc éolien Sud-1,wind,SUD1-WTG09,2023-12-13,modérée,9.7,7069.41,vent fort / tempête,TECH-SUD1-3,"En début d'après-midi, lors de l'inspection vi...",text,NaN
1,3,Centrale PV Est-2,solar,EST2-INV05,2023-08-27,mineure,1.0,261.57,temps calme,TECH-EST2-3,"En début d’après-midi, le système de supervisi...",text,NaN
2,4,Centrale PV Est-2,solar,EST2-INV01,2023-09-06,mineure,1.7,414.83,vent modéré,TECH-EST2-3,Lors d’un contrôle périodique sur la centrale ...,text,NaN
3,8,Centrale PV Est-2,solar,EST2-INV01,2023-07-05,modérée,14.8,5460.50,conditions nominales,TECH-EST2-2,"Vers la fin de l’après-midi, une alarme incend...",text,NaN
4,9,Centrale PV Est-2,solar,EST2-INV08,2024-06-22,modérée,4.7,1672.96,temps calme,TECH-EST2-2,"En début d'après-midi, une alarme a été déclen...",text,NaN


In [ ]:
EXTRACT_PROMPT = """Tu reçois l'image d'un rapport d'incident scanné.
Renvoie UNIQUEMENT un objet JSON avec exactement ces clés :
site, asset_type, equipment, date (format YYYY-MM-DD), severity,
downtime_hours (nombre), estimated_cost_eur (nombre), weather_condition,
technician, text (la description de l'incident, recopiée mot pour mot).
Si un champ est absent, mets null. N'invente rien."""


def pdf_to_png_b64(path: str, dpi: int = 150) -> list[str]:
    """Convertit chaque page du PDF en image PNG encodée en base64."""
    with pymupdf.open(path) as doc:
        return [base64.b64encode(p.get_pixmap(dpi=dpi).tobytes("png")).decode() for p in doc]


def extract_report(path: str, max_attempts: int = 8) -> dict:
    """OCR + extraction structurée d'un PDF scanné via le LLM vision."""
    content = [{"type": "text", "text": EXTRACT_PROMPT}] + [
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b}"}}
        for b in pdf_to_png_b64(path)
    ]
    for attempt in range(1, max_attempts + 1):
        try:
            r = client.with_options(max_retries=0).chat.completions.create(
                model=LLM_DEPLOYMENT,
                messages=[{"role": "user", "content": content}],
                response_format={"type": "json_object"},
            )
            return json.loads(r.choices[0].message.content)
        except RateLimitError as e:
            if attempt == max_attempts:
                raise
            m = re.search(r"retry after (\d+) seconds", str(e))
            time.sleep(int(m.group(1)) + 1 if m else 20)

In [7]:
scan_rows = df_all[df_all["document_format"] == "pdf_scan"]
test_path = scan_rows["document_path"].iloc[0]
print(test_path)
extract_report(test_path)

data/pdf_scan/incident_1.pdf


{'site': 'Centrale PV Ouest-1',
 'asset_type': 'Centrale photovoltaïque',
 'equipment': 'OUEST1-INV09',
 'date': '2024-12-20',
 'severity': 'modérée',
 'downtime_hours': 5.1,
 'estimated_cost_eur': 1843.26,
 'weather_condition': 'pluie modérée',
 'technician': 'TECH-OUEST1-3',
 'text': "Vers la fin de l'après-midi, un incendie s'est déclaré dans le bâtiment support de la centrale photovoltaïque, touchant une section de la toiture ainsi que plusieurs panneaux photovoltaïques. L'alerte a été déclenchée par le système de détection incendie intégré, qui a immédiatement alerté les équipes de sécurité et les pompiers. En réponse, une équipe de maintenance spécialisée a été dépêchée sur place pour assister les secours et sécuriser la zone, tandis que les pompiers ont rapidement engagé leur intervention. Après plusieurs heures d'efforts, l'incendie a été finalement maîtrisé par les pompiers, limitant les dégâts à une zone restreinte. L'incident a ainsi été considéré comme modéré, avec des domm

In [8]:
CACHE_PATH = "data/pdf_scan_extracted.json"
cache = json.load(open(CACHE_PATH, encoding="utf-8")) if os.path.exists(CACHE_PATH) else {}

scan_paths = scan_rows["document_path"].tolist()
todo = [p for p in scan_paths if p not in cache]
print(f"{len(todo)} PDFs à traiter ({len(cache)} déjà en cache)")


def work(p):
    try:
        return p, extract_report(p)
    except Exception as e:
        return p, {"error": str(e)}


def save_cache():
    with open(CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)


errors = 0
with cf.ThreadPoolExecutor(max_workers=4) as ex:  # baisse à 2 si trop de 429
    for i, (p, res) in enumerate(tqdm(ex.map(work, todo), total=len(todo))):
        if "error" in res:
            errors += 1
        else:
            cache[p] = res
        if (i + 1) % 20 == 0:
            save_cache()
save_cache()
print(f"Terminé. Erreurs : {errors} (relance la cellule pour les retenter)")

600 PDFs à traiter (0 déjà en cache)


100%|██████████| 600/600 [38:41<00:00,  3.87s/it]  

Terminé. Erreurs : 3 (relance la cellule pour les retenter)


In [28]:
all_scan = set(scan_rows["document_path"])

not_in_cache = sorted(all_scan - set(cache))
empty_text = [p for p, r in cache.items() if not (r.get("text") or "").strip()]
bad_fields = [p for p, r in cache.items()
              if r.get("severity") not in ("mineure", "modérée", "sévère", "moderee", "severe")
              or r.get("downtime_hours") is None or r.get("estimated_cost_eur") is None]

print("Absents du cache :", not_in_cache)
print("Texte vide       :", empty_text)
print("Champs douteux   :", bad_fields)

Absents du cache : ['data/pdf_scan/incident_1329.pdf', 'data/pdf_scan/incident_1685.pdf', 'data/pdf_scan/incident_1770.pdf']
Texte vide       : []
Champs douteux   : ['data/pdf_scan/incident_41.pdf', 'data/pdf_scan/incident_1225.pdf', 'data/pdf_scan/incident_1299.pdf', 'data/pdf_scan/incident_1361.pdf', 'data/pdf_scan/incident_1500.pdf', 'data/pdf_scan/incident_1763.pdf']


In [29]:
to_retry = not_in_cache + empty_text + ["data/pdf_scan/incident_1763.pdf"]

for p in dict.fromkeys(to_retry):          # sans doublons, ordre conservé
    _, res = work(p)
    if "error" in res:
        print("ÉCHEC", p, "->", res["error"][:200])
    else:
        cache[p] = res
        print("OK   ", p, "->", {k: res.get(k) for k in ["severity", "downtime_hours", "estimated_cost_eur"]})
save_cache()

OK    data/pdf_scan/incident_1329.pdf -> {'severity': 'modérée', 'downtime_hours': 11.5, 'estimated_cost_eur': 9191.25}
OK    data/pdf_scan/incident_1685.pdf -> {'severity': 'mineure', 'downtime_hours': 2.5, 'estimated_cost_eur': 1197.53}
OK    data/pdf_scan/incident_1770.pdf -> {'severity': 'mineure', 'downtime_hours': 4.9, 'estimated_cost_eur': 1019.96}
OK    data/pdf_scan/incident_1763.pdf -> {'severity': 'sévère', 'downtime_hours': 41.4, 'estimated_cost_eur': 52841.96}


In [30]:
import difflib
import unicodedata

# --- 1. Fusion (renommage des colonnes OCR vers les noms du CSV) ---
ext = pd.DataFrame.from_dict(cache, orient="index").rename_axis("document_path").reset_index()
ext = ext.rename(columns={"equipment": "equipment_id", "technician": "technician_id"})
ext = ext.drop(columns=["site"], errors="ignore")  # on garde le site du CSV

df_scan = (
    scan_rows.drop(columns=[c for c in ext.columns if c != "document_path"], errors="ignore")
    .merge(ext, on="document_path", how="inner")
)

# --- 2. Normalisation ---
def _strip(s: str) -> str:
    """minuscules + sans accents, pour comparer."""
    return "".join(
        c for c in unicodedata.normalize("NFD", str(s).lower()) if unicodedata.category(c) != "Mn"
    )

# asset_type : déduit du site (fiable) plutôt que de l'OCR
df_scan["asset_type"] = df_scan["site"].map(lambda s: "wind" if "olien" in _strip(s) else "solar")

# severity : rapprochement flou vers les 3 valeurs officielles
SEVERITIES = {"mineure": "mineure", "moderee": "modérée", "severe": "sévère"}  # clé sans accent -> valeur CSV

def norm_severity(v):
    if pd.isna(v):
        return None
    match = difflib.get_close_matches(_strip(v), SEVERITIES.keys(), n=1, cutoff=0.7)
    return SEVERITIES[match[0]] if match else None

df_scan["severity_raw"] = df_scan["severity"]
df_scan["severity"] = df_scan["severity_raw"].map(norm_severity)

# types numériques et date (même format que le CSV : YYYY-MM-DD)
for col in ["downtime_hours", "estimated_cost_eur"]:
    df_scan[col] = pd.to_numeric(df_scan[col], errors="coerce")
df_scan["date"] = pd.to_datetime(df_scan["date"], errors="coerce").dt.strftime("%Y-%m-%d")

# --- 3. Contrôles ---
unresolved = df_scan[df_scan["severity"].isna()]
print(f"{len(unresolved)} severity non résolues :")
print(unresolved[["document_path", "severity_raw"]].to_string())
print("\nValeurs manquantes par colonne (scans) :")
print(df_scan[["text", "date", "severity", "downtime_hours", "estimated_cost_eur"]].isna().sum())

# --- 4. Concaténation (les lignes 'text' restent en premier, même ordre qu'avant) ---
df_scan = df_scan.drop(columns=["severity_raw"])
df_text = df_all[df_all["document_format"] == "text"]
df = pd.concat([df_text, df_scan], ignore_index=True)
df = df[df["text"].notna() & (df["text"].str.strip() != "")].reset_index(drop=True)
print(df["document_format"].value_counts())
print(df["asset_type"].value_counts(), df["severity"].value_counts())

0 severity non résolues :
Empty DataFrame
Columns: [document_path, severity_raw]
Index: []

Valeurs manquantes par colonne (scans) :
text                  0
date                  0
severity              0
downtime_hours        0
estimated_cost_eur    0
dtype: int64
document_format
text        600
pdf_scan    600
Name: count, dtype: int64
asset_type
wind     600
solar    600
Name: count, dtype: int64 severity
mineure    714
modérée    357
sévère     129
Name: count, dtype: int64


In [15]:
import re
import unicodedata


def extract_pdf_text(pdf_path: str) -> str:
    """Texte de toutes les pages ; NFKC convertit les ligatures PDF (ﬁ → fi, ﬂ → fl)."""
    with pymupdf.open(pdf_path) as doc:
        text = "\n".join(page.get_text(sort=True) for page in doc).strip()
    return unicodedata.normalize("NFKC", text)


column_mapping = {
    "Site": "site",
    "Type d'installation": "asset_type",
    "Équipement": "equipment_id",
    "Date": "date",
    "Sévérité": "severity",
    "Durée d'arrêt (h)": "downtime_hours",
    "Coût estimé (€)": "estimated_cost_eur",
    "Conditions météo": "weather_condition",
    "Technicien": "technician_id",
}


def parse_incident_pdf(pdf_text: str) -> dict:
    """Sépare l'en-tête (champs) de la description, et extrait chaque champ."""
    result = {}
    parts = re.split(r"Description de l.incident", pdf_text, maxsplit=1, flags=re.IGNORECASE)
    header = parts[0]
    result["text"] = parts[1].strip() if len(parts) > 1 else ""

    for line in header.splitlines():
        line = line.strip()
        for label, col in column_mapping.items():
            if line.startswith(label):  # indépendant du nombre d'espaces
                value = line[len(label):].strip()
                if value:
                    result[col] = value
                break

    for col in ["downtime_hours", "estimated_cost_eur"]:
        if col in result:
            try:
                result[col] = float(result[col].replace(",", "."))
            except ValueError:
                result[col] = None
    return result

In [16]:
from tqdm import tqdm

df_pdf = df_all[df_all["document_format"] == "pdf_digital"].copy()
print("Nombre de PDF numériques :", len(df_pdf))


def read_pdf(path: str) -> str:
    try:
        return extract_pdf_text(path)
    except Exception as e:
        print(f"Erreur avec {path} : {e}")
        return ""


texts = [read_pdf(p) for p in tqdm(df_pdf["document_path"], desc="PDF digitaux")]
parsed = pd.DataFrame([parse_incident_pdf(t) for t in texts], index=df_pdf.index)

# Le texte vient du PDF ; les autres colonnes gardent la valeur du CSV si elle existe
df_pdf["text"] = parsed["text"]
for col in parsed.columns.drop("text"):
    if col in df_pdf.columns:
        df_pdf[col] = df_pdf[col].combine_first(parsed[col])
    else:
        df_pdf[col] = parsed[col]

df_pdf = df_pdf.reindex(columns=df_all.columns)

print(df_pdf[["site", "asset_type", "date", "severity", "downtime_hours", "estimated_cost_eur"]].isna().sum())
df_pdf.head()

Nombre de PDF numériques : 600


PDF digitaux: 100%|██████████| 600/600 [00:16<00:00, 35.61it/s]


site                  0
asset_type            0
date                  0
severity              0
downtime_hours        0
estimated_cost_eur    0
dtype: int64


,incident_id,site,asset_type,equipment_id,date,severity,downtime_hours,estimated_cost_eur,weather_condition,technician_id,text,document_format,document_path
5,6,Parc éolien Nord-1,Parc éolien,NORD1-WTG06,2024-05-21,mineure,3.2,1234.27,vent fort / tempête,TECH-NORD1-1,"Lors d'une tempête localisée, le système de pr...",pdf_digital,data/pdf_digital/incident_6.pdf
6,7,Centrale PV Est-1,Centrale photovoltaïque,EST1-INV11,2023-06-12,modérée,12.5,3852.80,temps calme,TECH-EST1-1,"Vers la fin de l’après-midi, un incendie s’est...",pdf_digital,data/pdf_digital/incident_7.pdf
10,11,Centrale PV Est-1,Centrale photovoltaïque,EST1-INV09,2025-07-21,modérée,12.9,4692.08,conditions nominales,TECH-EST1-2,"En début d’après-midi, un incendie s’est décla...",pdf_digital,data/pdf_digital/incident_11.pdf
13,14,Parc éolien Nord-1,Parc éolien,NORD1-WTG02,2025-09-07,sévère,55.8,70512.16,vent modéré,TECH-NORD1-3,"En début d'après-midi, une alarme incendie a é...",pdf_digital,data/pdf_digital/incident_14.pdf
17,18,Centrale PV Est-1,Centrale photovoltaïque,EST1-INV11,2024-02-17,mineure,2.0,415.87,pluie modérée,TECH-EST1-1,Lors d'une inspection de routine effectuée en ...,pdf_digital,data/pdf_digital/incident_18.pdf


In [17]:
import difflib


def _strip(s) -> str:
    """minuscules + sans accents, pour comparer."""
    return "".join(
        c for c in unicodedata.normalize("NFD", str(s).lower()) if unicodedata.category(c) != "Mn"
    )


SEVERITIES = {"mineure": "mineure", "moderee": "modérée", "severe": "sévère"}


def _norm_severity(v):
    if pd.isna(v):
        return None
    m = difflib.get_close_matches(_strip(v), SEVERITIES, n=1, cutoff=0.7)
    return SEVERITIES[m[0]] if m else None


def normalize_incidents(d: pd.DataFrame) -> pd.DataFrame:
    """Aligne asset_type / severity / types numériques / date sur le format du CSV."""
    d = d.copy()
    d["asset_type"] = d["site"].map(lambda s: "wind" if "olien" in _strip(s) else "solar")
    d["severity_raw"] = d["severity"]
    d["severity"] = d["severity_raw"].map(_norm_severity)
    for col in ["downtime_hours", "estimated_cost_eur"]:
        d[col] = pd.to_numeric(d[col], errors="coerce")
    d["date"] = pd.to_datetime(d["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    bad = d[d["severity"].isna()]
    if len(bad):
        print(f"{len(bad)} severity non résolues :")
        print(bad[["document_path", "severity_raw"]].to_string())
    return d.drop(columns="severity_raw")


# Fusion : les lignes 'text' restent en premier. df_scan est déjà normalisé (cellule scans).
df_text = df_all[df_all["document_format"] == "text"]
parts = [df_text, normalize_incidents(df_pdf)]
if "df_scan" in globals():
    parts.append(df_scan)

df = pd.concat(parts, ignore_index=True).reindex(columns=df_all.columns)
df = df[df["text"].notna() & (df["text"].str.strip() != "")].reset_index(drop=True)

print(df["document_format"].value_counts())
print(df["asset_type"].value_counts())
print(df["severity"].value_counts(dropna=False))
print(df[["date", "downtime_hours", "estimated_cost_eur"]].isna().sum())

document_format
text           600
pdf_digital    600
pdf_scan       597
Name: count, dtype: int64
asset_type
solar    899
wind     898
Name: count, dtype: int64
severity
mineure    1069
modérée     518
sévère      209
None          1
Name: count, dtype: int64
date                  0
downtime_hours        1
estimated_cost_eur    1
dtype: int64


In [19]:
for text in df["text"].sample(3, random_state=0):
    print(text)
    print("-" * 80)

Lors d’un contrôle périodique effectué dans la centrale photovoltaïque, une anomalie d'isolement a été détectée
sur une section de câblage située dans la zone des onduleurs. La détection a été réalisée par le système de
supervision lors de la vérification de routine, alertant l’équipe de maintenance. Rapidement, un technicien est
intervenu pour examiner la zone concernée et a confirmé la présence de cette anomalie d'isolement. La
correction a été effectuée sans incident ni blessé, en isolant la section défectueuse et en remplaçant le câblage
concerné. Après intervention, un nouvel essai d’isolement a montré que le problème était résolu. La ligne a été
remise en service et le contrôle a été clôturé.
--------------------------------------------------------------------------------
En début d’après-midi, une alarme incendie a été déclenchée dans le bâtiment support situé au cœur de la centrale photovoltaïque. La détection s’est produite suite à la surchauffe d’un ensemble de panneaux photo

## 3. Calculer les embeddings du texte

Un embedding représente un texte par un vecteur tel que deux textes similaires ont des vecteurs
proches. Calcul via Azure OpenAI, par lots, avec mise en cache locale (`.npy`) pour éviter de
refaire les appels à chaque exécution.

In [ ]:
import math
import re
import time

import numpy as np
from tqdm import tqdm


def wait_with_progress(seconds: int, desc: str) -> None:
    """Attente avec barre de progression (décompte en secondes)."""
    for _ in tqdm(range(seconds), desc=desc, unit="s", leave=False):
        time.sleep(1)


def embed_batch_with_retry(batch: list[str], max_attempts: int = 10):
    """Appelle l'API d'embeddings ; sur erreur 429, attend le délai indiqué par Azure puis réessaie.

    `max_retries=0` : on désactive les tentatives silencieuses du SDK pour que l'attente
    soit gérée ici, avec une barre de progression visible.
    """
    for attempt in range(1, max_attempts + 1):
        try:
            return client.with_options(max_retries=0).embeddings.create(
                model=EMBEDDING_DEPLOYMENT, input=batch
            )
        except RateLimitError as e:
            if attempt == max_attempts:
                raise
            match = re.search(r"retry after (\d+) seconds", str(e))
            wait = int(match.group(1)) + 1 if match else 30
            wait_with_progress(wait, f"Quota embeddings atteint (429), tentative {attempt}/{max_attempts}")


def azure_embed_documents(
    docs: list[str], batch_size: int = 16, partial_path: str = "embeddings_students.partial.npy"
) -> np.ndarray:
    """Calcule les embeddings par lots ; la progression est sauvegardée pour reprendre après interruption."""
    all_embeddings = list(np.load(partial_path)) if os.path.exists(partial_path) else []
    start = len(all_embeddings) // batch_size  # on ne garde que des lots complets
    all_embeddings = all_embeddings[: start * batch_size]
    n_batches = math.ceil(len(docs) / batch_size)

    for i in tqdm(range(start, n_batches), desc="Embeddings", unit="lot", initial=start, total=n_batches):
        batch = docs[i * batch_size : (i + 1) * batch_size]
        response = embed_batch_with_retry(batch)
        all_embeddings.extend(item.embedding for item in response.data)
        if (i + 1) % 10 == 0:
            np.save(partial_path, np.array(all_embeddings))

    if os.path.exists(partial_path):
        os.remove(partial_path)
    return np.array(all_embeddings)


'\nembeddings_path = "embeddings_students.npy"\nembeddings = None\nif os.path.exists(embeddings_path):\n    cached = np.load(embeddings_path)\n    if cached.ndim == 2 and cached.shape[0] == len(df):\n        embeddings = cached\n    else:\n        # Cache incohérent (calcul partiel, autre jeu de données...) : on le recalcule\n        print(f"Cache ignoré : forme {cached.shape}, attendu ({len(df)}, dim)")\n\nif embeddings is None:\n    embeddings = azure_embed_documents(df["text"].tolist())\n    np.save(embeddings_path, embeddings)\n\nassert embeddings.shape[0] == len(df), f"{embeddings.shape[0]} embeddings pour {len(df)} documents"\nembeddings.shape'

In [22]:
import pickle

CACHE_EMB = "embeddings_by_incident.pkl"
cache_emb = pickle.load(open(CACHE_EMB, "rb")) if os.path.exists(CACHE_EMB) else {}

# Amorçage : réutilise l'ancien cache des 600 rapports 'text' (même ordre que le CSV)
if not cache_emb and os.path.exists("embeddings_students.npy"):
    old = np.load("embeddings_students.npy")
    text_ids = df_all.loc[df_all["document_format"] == "text", "incident_id"].tolist()
    if old.shape[0] == len(text_ids):
        cache_emb = dict(zip(text_ids, old))
        print(f"{len(cache_emb)} embeddings récupérés de l'ancien cache")

# On ne calcule que ce qui manque
missing = df[~df["incident_id"].isin(cache_emb)]
print(f"{len(missing)} documents à encoder")
if len(missing):
    new = azure_embed_documents(missing["text"].tolist(), partial_path="embeddings_new.partial.npy")
    cache_emb.update(zip(missing["incident_id"], new))
    pickle.dump(cache_emb, open(CACHE_EMB, "wb"))

embeddings = np.array([cache_emb[i] for i in df["incident_id"]])
assert embeddings.shape[0] == len(df), f"{embeddings.shape[0]} embeddings pour {len(df)} documents"
embeddings.shape

600 embeddings récupérés de l'ancien cache
1197 documents à encoder


Embeddings: 100%|██████████| 75/75 [02:27<00:00,  1.96s/lot]


(1797, 1536)

## 4. Topic modeling avec BERTopic

- `CountVectorizer` : extrait les mots-clés de chaque topic une fois les clusters formés
  (stopwords français, tokens numériques exclus).
- `UMAP` : réduit la dimension des embeddings avant clustering.
- `HDBSCAN` : forme les clusters ; les documents isolés sont marqués `-1` (non classés).
- `min_cluster_size` : taille minimale d'un topic.

In [23]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

FRENCH_STOPWORDS = [
    "le",
    "la",
    "les",
    "un",
    "une",
    "des",
    "de",
    "du",
    "et",
    "en",
    "sur",
    "au",
    "aux",
    "pour",
    "par",
    "est",
    "sont",
    "a",
    "été",
    "suite",
    "lors",
    "dans",
    "que",
    "qui",
    "ce",
    "cette",
    "ces",
    "afin",
    "plus",
    "après",
    "avant",
]

umap_model = UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42
)

hdbscan_model = HDBSCAN(
    min_cluster_size=10,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)

vectorizer_model = CountVectorizer(
    stop_words=FRENCH_STOPWORDS,
    token_pattern=r"(?u)\b[^\W\d_][^\W\d_]+\b",
)

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(df["text"].tolist(), embeddings=embeddings)
df["topic"] = topics
print(f"{len(set(topics)) - (1 if -1 in topics else 0)} topics trouvés")

c:\Users\Julie Boyer\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-13 13:39:23,088 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-13 13:40:05,353 - BERTopic - Dimensionality - Completed ✓
2026-10-13 13:40:05,355 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-13 13:40:05,688 - BERTopic - Cluster - Completed ✓
2026-10-13 13:40:05,695 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-13 13:40:05,992 - BERTopic - Representation - Completed ✓


41 topics trouvés


## 5. Explorer les résultats

In [24]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,16,-1_onduleur_reports_panne_défaillance,"[onduleur, reports, panne, défaillance, succes...","[En début d’après-midi, une panne complète de ..."
1,0,261,0_vent_automatique_sécurité_arrêt,"[vent, automatique, sécurité, arrêt, vitesse, ...","[Lors d'une tempête, le système de sécurité du..."
2,1,100,1_chaud_point_boîtier_jonction,"[chaud, point, boîtier, jonction, thermographi...",[Lors d'une inspection thermographique program...
3,2,92,2_protection_surtension_ou_transitoire,"[protection, surtension, ou, transitoire, inte...","[Après un épisode météo intense, tel qu'une te..."
4,3,91,3_pale_vibrations_extrême_cosmétique,"[pale, vibrations, extrême, cosmétique, anorma...","[Lors d'une inspection de routine, une vibrati..."
5,4,90,4_isolement_câblage_périodique_correction,"[isolement, câblage, périodique, correction, e...",[Lors d’un contrôle périodique effectué sur le...
6,5,84,5_disjoncteur_armoire_nacelle_électrique,"[disjoncteur, armoire, nacelle, électrique, se...","[Vers la mi-journée, une anomalie électrique a..."
7,6,57,6_feu_extincteur_périmètre_départ,"[feu, extincteur, périmètre, départ, bâtiment,...","[Vers la fin de la matinée, une alarme incendi..."
8,7,56,7_prochaine_correction_mineure_mécanisme,"[prochaine, correction, mineure, mécanisme, an...",[Lors d'une inspection de routine sur le parc ...
9,8,51,8_connecteur_jonction_boîtier_préventif,"[connecteur, jonction, boîtier, préventif, dys...","[En début d’après-midi, une alerte a été génér..."


In [25]:
topic_model.visualize_barchart(top_n_topics=12)

In [26]:
topic_model.visualize_hierarchy()

Les mots-clés seuls suffisent rarement à comprendre un topic : regarde aussi
quelques documents représentatifs.

In [27]:
def show_topic_examples(topic_id: int, n: int = 3) -> None:
    print("Mots-clés :", [word for word, _ in topic_model.get_topic(topic_id)])
    examples = df.loc[df["topic"] == topic_id, "text"].head(n)
    for text in examples:
        print("-", text)


show_topic_examples(0)

Mots-clés : ['vent', 'automatique', 'sécurité', 'arrêt', 'vitesse', 'redémarrage', 'tempête', 'éoliennes', 'système', 'dépassant']
- En début d'après-midi, le système de contrôle a déclenché l'arrêt automatique de sécurité d'une éolienne située dans la zone dominante des vents forts. L'alerte a été immédiatement détectée par la supervision, qui a enregistré une montée rapide de la vitesse du vent dépassant la limite paramétrée. Un technicien a été dépêché sur place pour vérifier l’état de la machine et s’assurer qu’elle ne présente pas de dommages apparents. Après la fin de la tempête et la baisse du vent, le système de supervision a confirmé que la vitesse était revenue dans la plage normale, permettant ainsi le redémarrage automatique de l’éolienne. La machine a repris son fonctionnement en mode normal, sans intervention manuelle supplémentaire. Aucune anomalie majeure n’a été constatée lors de cette opération.
- Lors d'une journée venteuse, le système de sécurité du parc éolien a dé

## 6. Pistes pour aller plus loin

0. **Sauvegarder vos générations.** Pensez à sauvegarder vos générations pour éviter de perdre votre 
   travail et consommer des ressources LLM supplémentaires.
1. **Rajouter les docs PDF.** Lire et inclure les documents au format PDF digitaux et scans.
2. **Caractériser les topics.** Nommer les différents topics identifier et les analyser au mieux
    pour en faire ressortir les informations associées.
3. **Plan de remédiation.** Rédiger un plan de remédiation associé avec un agent
   ou un système d'agents en charge de differents aspects.
4. **Traiter les rapports `pdf_digital` et `pdf_scan`.** Ce starter kit ne couvre que le
   sous-ensemble `text` (600/1800 rapports).
5. **Séparer PV et éolien avant de clusteriser.** Le vocabulaire des deux domaines diffère
   fortement ; les mélanger risque de faire ressortir cette séparation avant les vraies causes.
6. **Ajuster `min_cluster_size`.** Plus petit → plus de topics, plus fins ; plus grand → moins de
   topics, plus larges.
7. **Ajuster `ngram_range`** pour capter les expressions à plusieurs mots.
8. **Fusionner les topics redondants** avec `topic_model.reduce_topics(...)`, guidé par
   `visualize_hierarchy()`.
9. **Croiser les topics avec les métadonnées** (`site`, `date`, `severity`, `downtime_hours`,
   `estimated_cost_eur`, `weather_condition`) pour enrichir l'interprétation.
10. **Approfondir avec une pipeline agentique.** Les deux appels LLM ci-dessus (nom + analyse)
   sont figés en un seul passage par topic. Un agent outillé (accès au dataframe, aux fonctions
   ci-dessus, à une recherche web) peut itérer pour approfondir l'analyse et croiser plusieurs
   topics entre eux, vérifier une hypothèse sur les métadonnées, ou creuser un signal faible avant
   de conclure, plutôt qu'une synthèse en un seul coup.
   